Author: Varun

In [3]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

### User input required
Put the data path on your system in the cell below

In [4]:
data_path = "/Users/varunpopli/Desktop/Data/CAR_-_EP_Flow_Activity_Queue__Agent_Names"

### User input ends

### Reading all filenames in the data folder

In [5]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


### Reading all data files
The code chunk below reads and appends all the CAR data files. The first two rows of each file are blank and thus ignored.

In [6]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)

1 CAR - EP, Flow, Activity, Queue, & Agent Names (01-12-25 - 01-18-25) (52010, 7)
2 CAR - EP, Flow, Activity, Queue, & Agent Names (01-19-25 - 02-01-25) (95495, 7)
3 CAR - EP, Flow, Activity, Queue, & Agent Names (02-02-25 - 02-15-25) (90056, 7)
4 CAR - EP, Flow, Activity, Queue, & Agent Names (02-16-25 - 03-01-25) (88186, 7)
5 CAR - EP, Flow, Activity, Queue, & Agent Names (03-02-25 - 03-15-25) (86377, 7)
6 CAR - EP, Flow, Activity, Queue, & Agent Names (04-07-24 - 04-20-24) (88766, 7)
7 CAR - EP, Flow, Activity, Queue, & Agent Names (04-21-24 - 05-04-24) (89643, 7)
8 CAR - EP, Flow, Activity, Queue, & Agent Names (05-05-24 - 05-18-24) (82575, 7)
9 CAR - EP, Flow, Activity, Queue, & Agent Names (05-19-24 - 06-01-24) (71103, 7)
10 CAR - EP, Flow, Activity, Queue, & Agent Names (06-02-24 - 06-15-24) (84354, 7)
11 CAR - EP, Flow, Activity, Queue, & Agent Names (06-16-24 - 06-29-24) (82124, 7)
12 CAR - EP, Flow, Activity, Queue, & Agent Names (06-30-24 - 07-13-24) (79752, 7)
13 CAR - EP, 

### Time datatype conversion
The code chunk below converts time from string to datetime datatype.

In [7]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [8]:
df_main.head()

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason
0,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,NaN,2025-01-14 12:39:32,NaN,NaN,NaN
1,001a3748-8d50-4550-8461-33547983deb0,NaN,LACMain,NaN,2025-01-14 12:39:32,NaN,NaN,NaN
2,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-01-14 12:39:32,NaN,NaN,NaN
3,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,LACMain,NaN,2025-01-14 12:39:32,NaN,NaN,NaN
4,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,MainMenu,2025-01-14 12:39:45,NaN,NaN,NaN


In [9]:
df_main.to_csv("combined_calls.csv", index=False)

In [17]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("combined_calls.csv")

# -----------------------------------
# 1. Normalize timestamps
# -----------------------------------
df["Date"] = pd.to_datetime(df["Activity Start Timestamp"], errors="coerce")
df["Year"]  = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"]   = df["Date"].dt.date


# -----------------------------------
# 2. MENU CLASSIFICATION (One Seniors Category)
# -----------------------------------
menu_patterns = {
    "Farmworker": r"Farmworker",
    "Seniors": r"Seniors",
    "HIVMenu": r"HIVMenu",
    "Family": r"Family",
    "Housing": r"Housing",
    "Benefits": r"Benefits",
    "Consumer": r"Consumer",
    "Criminal Records": r"Criminal Records",
    "Employment": r"Employment",
    "Immigration": r"Immigration",
    "ADAPT": r"ADAPT"
}

def classify_menu(activity_name):
    name = str(activity_name)
    for menu, pattern in menu_patterns.items():
        if re.search(pattern, name, re.IGNORECASE):
            return menu
    return "Other"

df["Menu_Category"] = df["Activity Name"].apply(classify_menu)


# ------------------------------------------------
# 3. SESSION-LEVEL SUBTYPE CLASSIFICATION
# ------------------------------------------------
# For each session, determine:
# - Was it agent answered?
# - Otherwise abandoned? voicemail? closed queue? etc.

session_subtype = []

for session_id, group in df.groupby("Contact Session ID"):
    activities = " ".join(group["Activity Name"].dropna().astype(str)).lower()
    agents = group["Agent Name"].dropna().astype(str)

    # Agent answered if ANY row has an agent name
    if any(a.strip() != "" for a in agents):
        subtype = "Agent Answered"
    elif "voicemail" in activities:
        subtype = "Voicemail"
    elif "closed queue" in activities or "closedqueue" in activities:
        subtype = "ClosedQueue"
    else:
        subtype = "Other"

    session_subtype.append([session_id, subtype])

session_subtype = pd.DataFrame(session_subtype, columns=["Contact Session ID", "Subtype"])


# Merge subtype back to the main dataframe
df = df.merge(session_subtype, on="Contact Session ID", how="left")


# ------------------------------------------------
# 4. DAILY UNIQUE SESSION COUNTS
# ------------------------------------------------
daily_counts = (
    df.groupby(["Menu_Category", "Subtype", "Year", "Month", "Day"])["Contact Session ID"]
      .nunique()
      .reset_index(name="Daily Unique Sessions")
)


# ------------------------------------------------
# 5. MONTHLY TOTALS + AVERAGE DAILY CALLS
# ------------------------------------------------
monthly_counts_new = (
    daily_counts.groupby(["Menu_Category", "Subtype", "Year", "Month"])
        .agg(
            Unique_Sessions_Per_Month=("Daily Unique Sessions", "sum"),
            Average_Daily_Unique_Calls=("Daily Unique Sessions", "mean")
        )
        .reset_index()
)

monthly_counts_new

/var/folders/qz/7779_hl55jj2kz5y73mhwl540000gn/T/ipykernel_56371/2214074661.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("combined_calls.csv")


,Menu_Category,Subtype,Year,Month,Unique_Sessions_Per_Month,Average_Daily_Unique_Calls
0,ADAPT,Agent Answered,2024,4,32,2.000000
1,ADAPT,Agent Answered,2024,5,46,2.705882
2,ADAPT,Agent Answered,2024,6,25,1.923077
3,ADAPT,Agent Answered,2024,7,25,1.562500
4,ADAPT,Agent Answered,2024,8,37,2.055556
...,...,...,...,...,...,...
601,Seniors,Voicemail,2024,8,1,1.000000
602,Seniors,Voicemail,2024,11,1,1.000000
603,Seniors,Voicemail,2024,12,1,1.000000
604,Seniors,Voicemail,2025,4,2,1.000000


In [18]:
monthly_counts_new.to_csv("combined_calls_transformed_new (1).csv", index=False)